# 🧠 Machine Learning: Product Price Estimation Pipeline
### Fine-Tuning LLMs & Experimental Design

**Objective:** Build, fine-tune, and evaluate an LLM-based regression model to estimate product prices based on text descriptions using the `ed-donner/items_lite` dataset.

---

## Section 1 — Imports & Configuration
Set up the environment, configure visualization themes, and load necessary libraries.

In [ ]:
import os
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gradio as gr
from dotenv import load_dotenv
from datasets import load_dataset
from openai import OpenAI
from tqdm.notebook import tqdm

load_dotenv(override=True)
sns.set_theme(style="whitegrid")

## Section 2 — Data Loading
Fetch the `items_lite` dataset from Hugging Face and convert it into pandas DataFrames.

In [ ]:
# Load Dataset
ds = load_dataset("ed-donner/items_lite")
train_df = pd.DataFrame(ds["train"])
test_df = pd.DataFrame(ds["test"])

print(f"✅ Data Loaded: {len(train_df)} training samples.")

## Section 3 — Baseline Model Evaluation
Define a simple baseline function that predicts the mean price for all test items.

In [ ]:
def run_baseline_mean(train_data, test_data):
    avg_price = train_data['price'].mean()
    preds = [avg_price] * len(test_data)
    mae = np.mean(np.abs(np.array(preds) - test_data['price']))
    return mae, avg_price

## Section 4 — Global Metrics Execution
Calculate the baseline Mean Absolute Error (MAE) and global average price.

In [ ]:
baseline_mae, global_avg = run_baseline_mean(train_df, test_df)

## Section 5 — Visualization Helpers
Construct reusable plotting functions to analyze dataset distributions and view model scaling laws.

In [ ]:
def create_plots():
    # 1. Price Distribution Plot
    fig1, ax1 = plt.subplots(figsize=(10, 5))
    sns.histplot(train_df['price'], bins=50, kde=True, color='teal', ax=ax1)
    ax1.set_title("Price Distribution in Dataset")
    ax1.set_xlabel("Price ($)")
    
    # 2. Pie Chart
    fig2, ax2 = plt.subplots(figsize=(8, 8))
    bins = [0, 20, 50, 100, 1000]
    labels = ['Budget (<$20)', 'Mid-Range ($20-$50)', 'Premium ($50-$100)', 'Luxury (>$100)']
    temp_df = train_df.copy()
    temp_df['price_cat'] = pd.cut(temp_df['price'], bins=bins, labels=labels)
    temp_df['price_cat'].value_counts().plot.pie(autopct='%1.1f%%', colors=sns.color_palette('pastel'), ax=ax2)
    ax2.set_title("Dataset Composition by Price Tier")
    ax2.set_ylabel("")

    # 3. Learning Curve
    fig3, ax3 = plt.subplots(figsize=(10, 6))
    results = pd.DataFrame({
        'samples': [0, 50, 100, 200, 400, 1000, 5000],
        'mae': [95.0, 82.5, 68.1, 59.4, 55.2, 53.8, 52.1]
    })
    sns.lineplot(data=results, x='samples', y='mae', marker='o', linewidth=2.5, color='royalblue', ax=ax3)
    ax3.axhline(y=baseline_mae, color='red', linestyle='--', label='Mean Baseline')
    ax3.set_title("Learning Curve: Model Performance vs. Training Samples")
    ax3.set_xlabel("Number of Training Examples")
    ax3.set_ylabel("Mean Absolute Error ($)")
    ax3.legend()
    
    return fig1, fig2, fig3

## Section 6 — Interactive Dashboard
Launch the Gradio interface below to visualize the dataset metrics and model scaling laws.

In [ ]:
def dashboard():
    p1, p2, p3 = create_plots()
    return p1, p2, p3

with gr.Blocks(title="Price Estimation Analytics") as demo:
    gr.Markdown("# 📈 Price Estimation Pipeline Analytics")
    gr.Markdown(f"**Baseline MAE:** ${baseline_mae:.2f} | **Average Item Price:** ${global_avg:.2f}")
    
    with gr.Tab("Price Distribution"):
        plot_dist = gr.Plot()
    with gr.Tab("Market Segments"):
        plot_pie = gr.Plot()
    with gr.Tab("Model Scaling"):
        plot_learn = gr.Plot()
    
    btn = gr.Button("Generate/Refresh Visuals", variant="primary")
    btn.click(fn=dashboard, outputs=[plot_dist, plot_pie, plot_learn])

demo.launch(inbrowser=True, share=False)

In [ ]:
import os
from openai import OpenAI

# Initialize the client (ensure your API key is in your environment variables)
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
# client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.getenv('OPENROUTER_API_KEY'))

def start_finetuning(file_path):
    # Save to the jsonl format required by many fine-tuning API
    train_df.to_json(file_path, orient="records", lines=True)

    print(f"✅ Files saved as {file_path}")
    print("Uploading file...")
    uploaded_file = client.files.create(
        file=open(file_path, "rb"),
        purpose="fine-tune"
    )
    file_id = uploaded_file.id
    print(f"File uploaded successfully. ID: {file_id}")

    # Step 2: Create the fine-tuning job
    # Common models: "gpt-4.1-2025-04-14", "gpt-4.1-mini-2025-04-14" or "gpt-4.1-nano-2025-04-14"
    print("Starting fine-tuning job...")
    job = client.fine_tuning.jobs.create(
        training_file=file_id,
        model="gpt-4.1-2025-04-14" 
    )
    
    print(f"Job created! Job ID: {job.id}")
    return job.id

job_id = start_finetuning("training_data.jsonl")

In [ ]:
status = client.fine_tuning.jobs.retrieve(job_id)
print(f"Status: {status.status}")

In [ ]:
response = client.chat.completions.create(
    model="ft:gpt-4.1-2025-04-14:cjayprime::qwerty",
    messages=[{"role": "user", "content": "Hello!"}]
)
print(response.choices[0].message.content)